Method 1: Sql 

In [0]:
%sql
create or replace table retail_products_sql(
    product_id int,
    product_name string,
    price int
)
using Delta

In [0]:
%sql
insert into retail_products_sql values
(1,'Laptop',50000),
(2,'Mouse',800),
(3,'Keyboard',1500);

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
select *
from retail_products_sql;

product_id,product_name,price
1,Laptop,50000
2,Mouse,800
3,Keyboard,1500


Method 2: Dataframes

In [0]:
data = [
    (4, "Monitor", 12000),
    (5, "Printer", 18000),
    (6, "Webcam", 2500)
]
 
df = spark.createDataFrame(
    data,
    ["product_id", "product_name", "price"]
)
 
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail_products_pyspark")
 
display(spark.table("retail_products_pyspark"))
 

product_id,product_name,price
4,Monitor,12000
5,Printer,18000
6,Webcam,2500


Method 3: using the query (CTAS) - Create Table As Select

In [0]:
%sql
CREATE OR REPLACE TABLE retail_products_ctas
USING DELTA
AS
SELECT *
FROM VALUES
    (7, 'Tablet', 25000),
    (8, 'Speaker', 5000),
    (9, 'Router', 3500)
AS products(product_id, product_name, price);

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM retail_products_ctas;

product_id,product_name,price
7,Tablet,25000
8,Speaker,5000
9,Router,3500


Merge and Upsert

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW product_updates AS
SELECT *
FROM VALUES
    (2, 'Wireless Mouse', 950),
    (10, 'USB Hub', 1200)
AS updates(product_id, product_name, price);

In [0]:
%sql
MERGE INTO retail_products_sql AS target
USING product_updates AS source
ON target.product_id = source.product_id

WHEN MATCHED THEN
    UPDATE SET
        target.product_name = source.product_name,
        target.price = source.price

WHEN NOT MATCHED THEN
    INSERT (product_id, product_name, price)
    VALUES (source.product_id, source.product_name, source.price);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


In [0]:
%sql
SELECT * FROM retail_products_sql
ORDER BY product_id;

product_id,product_name,price
1,Laptop,50000
2,Wireless Mouse,950
3,Keyboard,1500
10,USB Hub,1200


In [0]:
%sql
describe detail retail_products_sql;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,915fb3bb-67d8-4d08-929f-dc732276fe40,etl_training.default.retail_products_sql,null,abfss://unity-catalog-storage@dbstoragelolvtq6loxhne.dfs.core.windows.net/7405609821935979/__unitystorage/catalogs/a3c11d72-07c4-4178-94cc-f42f4ea63830/tables/89e9dc56-73d1-4ff0-9161-a8f9f5419ee5,2026-11-05T07:11:44.180Z,2026-11-05T08:49:11.000Z,List(),List(),1,2214,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> 89e9dc56-73d1-4ff0-9161-a8f9f5419ee5, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-3056597f-f54e-4354-a480-cdb0911f6e9d, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-36ef68e2-6177-43b1-b20b-ac38a7decd30, delta.enableInCommitTimestamps -> true)",3,7,"List(appendOnly, catalogManaged, clustering, deletionVectors, domainMetadata, inCommitTimestamp, invariants, rowTracking, v2Checkpoint, vacuumProtocolCheck)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",true


In [0]:
%sql
describe history retail_products_sql

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-11-05T08:49:11.855Z,145401548560959,azuser7402_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(4421094106350401),538f3955-de94-46ba-a77d-7c88c48f6d9f,1105-084807-qxqkf2s8-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 5326, p25FileSize -> 2214, numDeletionVectorsRemoved -> 1, minFileSize -> 2214, numAddedFiles -> 1, maxFileSize -> 2214, p75FileSize -> 2214, p50FileSize -> 2214, numAddedBytes -> 2214)",null,Databricks-Runtime/19.13.x-photon-scala2.13
2,2026-11-05T08:49:06.978Z,145401548560959,azuser7402_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(product_id#10124 = product_id#10115)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(4421094106350401),538f3955-de94-46ba-a77d-7c88c48f6d9f,1105-084807-qxqkf2s8-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 4009, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 10640, materializeSourceTimeMs -> 871, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 5128, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 4395)",null,Databricks-Runtime/19.13.x-photon-scala2.13
1,2026-11-05T07:12:40.483Z,145401548560959,azuser7402_mml.local@karthikirisoutlook.onmicrosoft.com,WRITE,"Map(partitionBy -> [], clusterBy -> [], statsOnLoad -> true, mode -> Append, clusteringOnWriteStatus -> null)",null,List(4421094106350401),88fcfd1c-5783-44b1-9ad4-f710210d2136,1105-071128-6wm4pf3g-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1317)",null,Databricks-Runtime/19.13.x-photon-scala2.13
0,2026-11-05T07:11:44.960Z,145401548560959,azuser7402_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""databricks.internal.autoUpgrades.delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""89e9dc56-73d1-4ff0-9161-a8f9f5419ee5"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-3056597f-f54e-4354-a480-cdb0911f6e9d"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-36ef68e2-6177-43b1-b20b-ac38a7decd30"",""databricks.internal.autoUpgrades.delta.feature.catalogManaged"":""supported""}, statsOnLoad -> false, clusterByAuto -> true)",null,List(4421094106350401),98872579-32f2-4c76-a145-f82bf3c703ce,1105-071128-6wm4pf3g-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.13.x-photon-scala2.13


In [0]:
%sql
select *
from retail_products_sql version as of 1;


product_id,product_name,price
1,Laptop,50000
2,Mouse,800
3,Keyboard,1500


In [0]:
%sql
select * 
from retail_products_sql timestamp as of '2026-11-05T08:49:06.978+00:00'

product_id,product_name,price
2,Wireless Mouse,950
10,USB Hub,1200
1,Laptop,50000
3,Keyboard,1500


In [0]:
%sql
optimize retail_products_sql;

path,metrics
abfss://unity-catalog-storage@dbstoragelolvtq6loxhne.dfs.core.windows.net/7405609821935979/__unitystorage/catalogs/a3c11d72-07c4-4178-94cc-f42f4ea63830/tables/89e9dc56-73d1-4ff0-9161-a8f9f5419ee5,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1793869623797, 1793869624712, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null, 0)"


Zorder - to group relevant thing as single unit

In [0]:
%sql
optimize retail_products_sql
zorder by (product_id);

path,metrics
abfss://unity-catalog-storage@dbstoragelolvtq6loxhne.dfs.core.windows.net/7405609821935979/__unitystorage/catalogs/a3c11d72-07c4-4178-94cc-f42f4ea63830/tables/89e9dc56-73d1-4ff0-9161-a8f9f5419ee5,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 2214), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1793870020359, 1793870021202, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null, 0)"
